# Upload das classificadas 2025 (Drive → Google Cloud Storage)

Sobe os GeoTIFFs classificados de **2025** que estão no Google Drive para um bucket
do Google Cloud Storage — mesma ideia do `uploadTIF_from_localFolder_GoogleStorage.py`,
mas lendo do Drive montado no Colab.

- **Entrada:** `/content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025/`
- **Padrão de nome (outro prefixo):** `<region_id>_2025_pred.tif`  (sufixo `_pred`,
  diferente do `pred_<region>_<ano>.tif` do fluxo antigo)
- **Destino:** `gs://<BUCKET>/<GCS_PREFIX>/<FOLDER>/<region_id>_2025_pred.tif`

Cada TIF é convertido para COG (Cloud Optimized GeoTIFF) antes do envio, igual ao script local.

## Célula 1 — Instalar dependências e montar o Drive

In [ ]:
# google-cloud-storage para o upload; gdal-bin traz o gdal_translate (CLI),
# que NÃO vem por padrão no Colab (era a causa do erro 32512 = 127 'command not found').
!pip -q install google-cloud-storage
!apt-get -qq install -y gdal-bin > /dev/null

# Confere o que ficou disponível (qualquer um dos dois serve p/ gerar COG):
!gdal_translate --version || echo 'gdal_translate CLI indisponível'
try:
    from osgeo import gdal as _gdal
    print('osgeo.gdal', _gdal.__version__)
except Exception as _e:
    print('osgeo.gdal indisponível:', _e)

from google.colab import drive
drive.mount('/content/drive')

## Célula 2 — Parâmetros de entrada
Edite antes de rodar. Os defaults seguem o script local (`mapbiomas-energia` / `mapbiomas-agua`).

In [ ]:
# ── Fonte no Drive ───────────────────────────────────────────────────────
# Classificadas: .../DL_Fotovoltaica/tif_classificadas_2025  (arquivos *_2025_pred.tif)
# Reduzidas    : .../DL_Fotovoltaica/tif_reduzidas_2025       (arquivos *_2025_img_reduzido.tif)
TIF_DIR      = '/content/drive/MyDrive/DL_Fotovoltaica/tif_reduzidas_2025'

# Só arquivos que casam com este glob sobem (ignora subpastas, CSVs e .log).
# Aceita qualquer sufixo: *_2025_pred.tif, *_2025_img_reduzido.tif, etc.
GLOB_PATTERN = '*_2025*.tif'

# ── Destino no Google Cloud Storage ──────────────────────────────────────
BUCKET       = 'mapbiomas-energia'
GCS_PREFIX   = 'fotovoltaicas_tif'          # pasta raiz dentro do bucket
FOLDER       = 'classificadas_2025'         # subpasta (nome curto do modelo/rodada)
PROJECT      = 'mapbiomas-agua'

# Chave da service account. Suba o JSON para o Drive e aponte aqui.
# Se deixar None, cai no login interativo do Colab (google.colab.auth).
KEY_JSON     = '/content/drive/MyDrive/DL_Fotovoltaica/keys/mapbiomas-agua-36521f541610.json'

# ── Filtros opcionais ────────────────────────────────────────────────────
YEARS        = [2025]      # None = todos os anos encontrados
REGIONS      = None        # ex.: ['00000000000000000b3c'] ; None = todas
MAKE_COG     = False       # False = sobe o TIF original sem converter (True = gera COG)
OVERWRITE    = False       # False = pula blobs que já existem no bucket

print('Parâmetros carregados.')
print(f'  Fonte  : {TIF_DIR}  (glob={GLOB_PATTERN})')
print(f'  Destino: gs://{BUCKET}/{GCS_PREFIX}/{FOLDER}')
print(f'  Anos={YEARS} | Regiões={REGIONS} | COG={MAKE_COG} | Overwrite={OVERWRITE}')

## Célula 3 — Imports e helpers

In [ ]:
import os
import re
import tempfile
from pathlib import Path

from google.cloud import storage

# Agnóstico ao sufixo: <region_id>_<year>[_qualquer].tif
# Ex.: <region>_2025_pred.tif  ou  <region>_2025_img_reduzido.tif
_FNAME_RE = re.compile(r'^(.+?)_(\d{4})(?:_.*)?\.tif$')


def parse_name(fname: str):
    """Extrai (region_id, year) de <region>_<year>[_sufixo].tif ; None se não casar."""
    m = _FNAME_RE.match(fname)
    if not m:
        return None
    return m.group(1), int(m.group(2))


def to_cog(src: Path, dst: Path):
    """Converte para Cloud Optimized GeoTIFF.

    Tenta primeiro a API Python (osgeo.gdal) e, se indisponível, o gdal_translate (CLI).
    """
    # 1) osgeo.gdal (não depende do binário estar no PATH)
    try:
        from osgeo import gdal
        gdal.UseExceptions()
        gdal.Translate(
            str(dst), str(src),
            format='COG',
            creationOptions=['COMPRESS=LZW', 'BIGTIFF=IF_SAFER'],
        )
        return
    except ImportError:
        pass  # sem bindings Python; tenta o CLI abaixo

    # 2) gdal_translate (CLI) — precisa do gdal-bin (Célula 1)
    cmd = (
        f'gdal_translate "{src}" "{dst}" '
        f'-co TILED=YES -co COPY_SRC_OVERVIEWS=YES -co COMPRESS=LZW'
    )
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError(
            f'Falha ao gerar COG (código {ret}) para {src}. '
            f'Rode a Célula 1 (instala gdal-bin) ou defina MAKE_COG=False.'
        )


def make_client() -> storage.Client:
    """Cliente GCS: usa a service account se KEY_JSON existir, senão auth do Colab."""
    if KEY_JSON and Path(KEY_JSON).exists():
        print(f'Autenticando via service account: {KEY_JSON}')
        return storage.Client.from_service_account_json(KEY_JSON, project=PROJECT)
    print('KEY_JSON ausente — usando login interativo do Colab.')
    from google.colab import auth
    auth.authenticate_user()
    return storage.Client(project=PROJECT)


print('Helpers prontos.')

## Célula 4 — Upload (loop principal)

In [ ]:
tif_dir = Path(TIF_DIR)
tif_files = sorted(tif_dir.glob(GLOB_PATTERN))
print("numero de imagens carregadas ", len(tif_files))
if not tif_files:
    raise FileNotFoundError(f'Nenhum arquivo {GLOB_PATTERN!r} em {tif_dir}')

client = make_client()
bucket = client.bucket(BUCKET)

years_set   = set(YEARS)   if YEARS   else None
regions_set = set(REGIONS) if REGIONS else None

print('=' * 60)
print(f'Fonte  : {tif_dir}  ({len(tif_files)} arquivos casam o glob)')
print(f'Destino: gs://{BUCKET}/{GCS_PREFIX}/{FOLDER}')
print(f'Anos={YEARS or "todos"} | Regiões={REGIONS or "todas"} | COG={MAKE_COG}')
print('=' * 60)

enviados = pulados = erros = 0
with tempfile.TemporaryDirectory(prefix='cog_tmp_') as tmpdir:
    for i, tif_path in enumerate(tif_files, 1):
        parsed = parse_name(tif_path.name)
        if not parsed:
            print(f'[{i}/{len(tif_files)}] fora do padrão, pulando: {tif_path.name}')
            continue
        region_id, year = parsed

        if years_set   is not None and year      not in years_set:
            continue
        if regions_set is not None and region_id not in regions_set:
            continue

        blob_name = f'{GCS_PREFIX}/{FOLDER}/{tif_path.name}'
        blob = bucket.blob(blob_name)

        if not OVERWRITE and blob.exists():
            print(f'[{i}/{len(tif_files)}] já existe, pulando: gs://{BUCKET}/{blob_name}')
            pulados += 1
            continue

        print(f'[{i}/{len(tif_files)}] {tif_path.name}  (region={region_id} year={year})')
        cog_path = Path(tmpdir) / tif_path.name
        try:
            if MAKE_COG:
                to_cog(tif_path, cog_path)
                src_upload = cog_path
            else:
                src_upload = tif_path
            blob.upload_from_filename(str(src_upload))
            print(f'  upload OK → gs://{BUCKET}/{blob_name}')
            enviados += 1
        except Exception as exc:
            print(f'  Erro em {tif_path.name}: {exc}')
            erros += 1
        finally:
            if cog_path.exists():
                cog_path.unlink()

print('=' * 60)
print(f'Concluído. Enviados={enviados} | Pulados={pulados} | Erros={erros}')